### Hyperparameter Tuning and Training for Explainable Boosting Machines (EBM)

This notebook performs hyperparameter tuning for EBMs using interpretML package.


In [1]:
import numpy as np
import pandas as pd
import json
import time
import sys
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, mean_squared_error, mean_absolute_error
from interpret.glassbox import ExplainableBoostingClassifier, ExplainableBoostingRegressor

# Add src to path for imports
project_root = Path.cwd().parent.parent
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import neural_additive_models.data_utils as data_utils
import warnings
warnings.filterwarnings('ignore')


## Configuration


In [2]:
# Dataset configuration
dataset_name = 'OpenML_45402_regression'  # Change this to your dataset
is_regression = True  # Set to True for regression

# Hyperparameter search space
hp_search_space = {
    'learning_rate': [0.001, 0.1],
    'max_bins': [16, 32, 64, 128, 256, 512],
    'max_interaction_bins': [8, 16, 32, 64, 128],
    'interactions': [0, 2, 5, 10, 15, 20],
    'outer_bags': [1, 2, 4, 8, 16],
    'inner_bags': [0, 2, 4, 8, 16],
    'min_samples_leaf': [1, 2, 3, 5, 10],
    'max_leaves': [2, 3, 5, 10, 15, 20]
}

# Fixed hyperparameters
fixed_hp = {
    'random_state': 42,
    'n_jobs': -1,
    'early_stopping_rounds': 50,
    'validation_size': 0.125
}

# Tuning parameters
n_trials = 50
random_seed = 42

# Set results directory
project_root = Path.cwd().parent.parent
results_dir = project_root / 'results' / 'hyperparameter_tuning' / 'ebm'
results_dir.mkdir(parents=True, exist_ok=True)


## Load Dataset


In [3]:
# Load dataset
print(f"Loading dataset: {dataset_name}")
data_x, data_y, column_names = data_utils.load_dataset(dataset_name)

# Determine if regression from dataset name
if '_regression' in dataset_name:
    is_regression = True
elif '_classification' in dataset_name:
    is_regression = False

print(f"Dataset shape: {data_x.shape}")
print(f"Target shape: {data_y.shape}")
print(f"Number of features: {data_x.shape[1]}")
print(f"Task type: {'Regression' if is_regression else 'Classification'}")

# Split into train/val/test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    data_x, data_y, test_size=0.2, random_state=random_seed,
    stratify=data_y if not is_regression else None
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.2, random_state=random_seed,
    stratify=y_train_val if not is_regression else None
)

print(f"Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")


Loading dataset: OpenML_45402_regression
Dataset shape: (3107, 6)
Target shape: (3107,)
Number of features: 6
Task type: Regression
Train: 1988, Val: 497, Test: 622


## Hyperparameter Sampling


In [4]:
def sample_hyperparameters(search_space, random_seed=None):
    """Sample hyperparameters from search space."""
    np.random.seed(random_seed)
    hp = {}
    
    for key, values in search_space.items():
        if isinstance(values, list) and len(values) == 2 and all(isinstance(x, (int, float)) for x in values):
            # Continuous range
            hp[key] = float(np.random.uniform(values[0], values[1]))
        else:
            # Discrete choices
            value = np.random.choice(values)
            hp[key] = value.item() if isinstance(value, np.generic) else value
    
    return hp

# Generate hyperparameter configurations
hyperparameters = []
for trial in range(n_trials):
    trial_seed = random_seed + trial
    hp_config = sample_hyperparameters(hp_search_space, trial_seed)
    hyperparameters.append({
        'trial': trial + 1,
        'hyperparameters': hp_config
    })

print(f"Generated {n_trials} hyperparameter configurations")


Generated 50 hyperparameter configurations


## Hyperparameter Tuning


In [5]:
def train_and_evaluate_ebm(X_train, y_train, X_val, y_val, hyperparameters, is_regression=False):
    """Train EBM and return validation score."""
    
    if is_regression:
        model = ExplainableBoostingRegressor(
            **hyperparameters,
            **fixed_hp
        )
    else:
        model = ExplainableBoostingClassifier(
            **hyperparameters,
            **fixed_hp
        )
    
    # Train
    start_time = time.time()
    model.fit(X_train, y_train)
    training_time = time.time() - start_time
    
    # Predict and evaluate
    y_pred = model.predict(X_val)
    
    if is_regression:
        # RMSE for regression
        score = np.sqrt(mean_squared_error(y_val, y_pred))
    else:
        # AUC for classification
        y_pred_proba = model.predict_proba(X_val)[:, 1]
        score = roc_auc_score(y_val, y_pred_proba)
    
    return score, training_time, model

# Run hyperparameter tuning
print("="*70)
print(f"HYPERPARAMETER TUNING - {n_trials} trials")
print("="*70)

trial_results = []

for trial_data in hyperparameters:
    trial_num = trial_data['trial']
    hp = trial_data['hyperparameters']
    
    print(f"\nTrial {trial_num}/{n_trials}...", end=' ', flush=True)
    
    try:
        score, train_time, model = train_and_evaluate_ebm(
            X_train, y_train, X_val, y_val, hp, is_regression
        )
        
        metric_name = 'RMSE' if is_regression else 'AUC'
        print(f"{metric_name}: {score:.4f} ({train_time:.1f}s)")
        
        trial_results.append({
            'trial': trial_num,
            'hyperparameters': hp,
            'validation_score': score,
            'training_time': train_time,
            'success': True
        })
    except Exception as e:
        print(f"Failed: {str(e)[:50]}")
        trial_results.append({
            'trial': trial_num,
            'hyperparameters': hp,
            'validation_score': None,
            'training_time': None,
            'success': False,
            'error': str(e)
        })

print("\n" + "="*70)
print("TUNING COMPLETE")
print("="*70)


HYPERPARAMETER TUNING - 50 trials

Trial 1/50... RMSE: 0.1100 (5.4s)

Trial 2/50... RMSE: 0.1235 (0.5s)

Trial 3/50... RMSE: 0.1099 (10.3s)

Trial 4/50... RMSE: 0.1154 (0.7s)

Trial 5/50... RMSE: 0.1169 (21.7s)

Trial 6/50... RMSE: 0.1248 (0.2s)

Trial 7/50... RMSE: 0.1173 (4.1s)

Trial 8/50... RMSE: 0.1131 (485.8s)

Trial 9/50... RMSE: 0.1158 (1.4s)

Trial 10/50... RMSE: 0.1188 (3.0s)

Trial 11/50... RMSE: 0.1159 (1.8s)

Trial 12/50... RMSE: 0.1080 (35.8s)

Trial 13/50... RMSE: 0.1200 (0.1s)

Trial 14/50... RMSE: 0.1198 (4.0s)

Trial 15/50... RMSE: 0.1129 (0.8s)

Trial 16/50... RMSE: 0.1188 (0.2s)

Trial 17/50... RMSE: 0.1224 (2.7s)

Trial 18/50... RMSE: 0.1006 (4.5s)

Trial 19/50... RMSE: 0.1202 (0.1s)

Trial 20/50... RMSE: 0.1169 (0.5s)

Trial 21/50... RMSE: 0.1174 (5.1s)

Trial 22/50... RMSE: 0.1158 (0.1s)

Trial 23/50... RMSE: 0.1174 (93.1s)

Trial 24/50... RMSE: 0.1150 (1.0s)

Trial 25/50... RMSE: 0.1187 (15.4s)

Trial 26/50... RMSE: 0.1168 (1.9s)

Trial 27/50... RMSE: 0.1172 (4.

In [6]:
df_results = pd.DataFrame(trial_results)

# Filter successful trials
df_success = df_results[df_results['success']].copy()

if len(df_success) > 0:
    # Find best hyperparameters
    if is_regression:
        # Lower is better for RMSE
        best_idx = df_success['validation_score'].idxmin()
    else:
        # Higher is better for AUC
        best_idx = df_success['validation_score'].idxmax()
    
    best_trial = df_success.loc[best_idx]
    
    print(f"Best trial: {int(best_trial['trial'])}")
    metric_name = 'RMSE' if is_regression else 'AUC'
    print(f"Best {metric_name}: {best_trial['validation_score']:.4f}")
    print(f"Training time: {best_trial['training_time']:.1f}s")
    print("\nBest hyperparameters:")
    for key, value in best_trial['hyperparameters'].items():
        print(f"  {key}: {value}")
    
    # Save best hyperparameters
    best_hp_file = results_dir / f"best_hp_{dataset_name.replace('/', '_').replace(':', '_')}.json"
    with open(best_hp_file, 'w') as f:
        json.dump(best_trial['hyperparameters'], f, indent=2)
    print(f"\nSaved best hyperparameters to: {best_hp_file}")
    
    # Save all results
    results_file = results_dir / f"tuning_results_{dataset_name.replace('/', '_').replace(':', '_')}.json"
    with open(results_file, 'w') as f:
        json.dump(trial_results, f, indent=2)
    print(f"Saved all results to: {results_file}")
    
    # Display summary statistics
    print("\n" + "="*70)
    print("SUMMARY STATISTICS")
    print("="*70)
    print(f"Successful trials: {len(df_success)}/{n_trials}")
    print(f"Mean {metric_name}: {df_success['validation_score'].mean():.4f}")
    print(f"Std {metric_name}: {df_success['validation_score'].std():.4f}")
    print(f"Mean training time: {df_success['training_time'].mean():.1f}s")
else:
    print("No successful trials!")


Best trial: 18
Best RMSE: 0.1006
Training time: 4.5s

Best hyperparameters:
  learning_rate: 0.09247947644186812
  max_bins: 512
  max_interaction_bins: 64
  interactions: 20
  outer_bags: 2
  inner_bags: 0
  min_samples_leaf: 5
  max_leaves: 2

Saved best hyperparameters to: c:\Users\dejvi\Documents\pythonProject\neural-additive-models-xai-seminar-1\results\hyperparameter_tuning\ebm\best_hp_OpenML_45402_regression.json
Saved all results to: c:\Users\dejvi\Documents\pythonProject\neural-additive-models-xai-seminar-1\results\hyperparameter_tuning\ebm\tuning_results_OpenML_45402_regression.json

SUMMARY STATISTICS
Successful trials: 50/50
Mean RMSE: 0.1150
Std RMSE: 0.0059
Mean training time: 26.9s
